In [1]:
# Standard libraries
import pandas as pd
import numpy as np
import random

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem import rdFingerprintGenerator

# Machine Learning Models
from sklearn.ensemble import RandomForestRegressor

import torch
import torch_geometric
from torch_geometric.data import Data

# ML Metrics
from sklearn.metrics import mean_absolute_error

from sklearn.model_selection import train_test_split

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Geometric version: {torch_geometric.__version__}")

PyTorch version: 2.7.1+cu118
PyTorch Geometric version: 2.7.0


## Import Dataset

In [2]:
df = pd.read_csv(r'D:\Yan\Repo\GitHub\MolecuGraph\data\qm9.csv')
df.head(5)

,mol_id,smiles,A,B,C,mu,alpha,homo,lumo,gap,...,zpve,u0,u298,h298,g298,cv,u0_atom,u298_atom,h298_atom,g298_atom
0,gdb_1,C,157.71180,157.709970,157.706990,0.0000,13.21,-0.3877,0.1171,0.5048,...,0.044749,-40.478930,-40.476062,-40.475117,-40.498597,6.469,-395.999595,-398.643290,-401.014647,-372.471772
1,gdb_2,N,293.60975,293.541110,191.393970,1.6256,9.46,-0.2570,0.0829,0.3399,...,0.034358,-56.525887,-56.523026,-56.522082,-56.544961,6.316,-276.861363,-278.620271,-280.399259,-259.338802
2,gdb_3,O,799.58812,437.903860,282.945450,1.8511,6.31,-0.2928,0.0687,0.3615,...,0.021375,-76.404702,-76.401867,-76.400922,-76.422349,6.002,-213.087624,-213.974294,-215.159658,-201.407171
3,gdb_4,C#C,0.00000,35.610036,35.610036,0.0000,16.28,-0.2845,0.0506,0.3351,...,0.026841,-77.308427,-77.305527,-77.304583,-77.327429,8.574,-385.501997,-387.237686,-389.016047,-365.800724
4,gdb_5,C#N,0.00000,44.593883,44.593883,2.8937,12.99,-0.3604,0.0191,0.3796,...,0.016601,-93.411888,-93.409370,-93.408425,-93.431246,6.278,-301.820534,-302.906752,-304.091489,-288.720028


## Data Preparation

In [3]:
df_subset = df.head(5000).copy()
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_fp(smiles):
    """Generate a Morgan Fingerprint with radius 2 and 1024 bits"""
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        fp = mfpgen.GetFingerprint(mol)
        return np.array(list(fp.ToBitString())).astype(int)
    else:
        return np.zeros(1024) # Return empty if RDKit can't read it


X = np.array([smiles_to_fp(s) for s in df_subset['smiles']])
y = df_subset['gap'].values

## Train and Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and Train
model = RandomForestRegressor(n_estimators=100, n_jobs=-1)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

## Evaluate

In [5]:
pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)

print(f"Baseline MAE: {mae:.4f} eV")

Baseline MAE: 0.0116 eV


In [6]:
# Create a tiny "toy" graph to test the structure
# Atoms: 3, Edges: 2 (Atom 0 -> 1, 1 -> 2)
edge_index = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)
x = torch.tensor([[1], [1], [1]], dtype=torch.float)
data = Data(x=x, edge_index=edge_index)

print(f"\nSuccessfully created a graph object: {data}")


Successfully created a graph object: Data(x=[3, 1], edge_index=[2, 4])


In [7]:
# Check if CUDA (NVIDIA GPU acceleration) is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: NVIDIA GeForce GTX 1650


In [9]:
from src.model import MoleculeGNN

# Initialize model
model = MoleculeGNN(num_node_features=10, hidden_channels=128) # 10 is a placeholder for now
model.to(device) # Move to GPU!
print(model)

MoleculeGNN(
  (conv1): GCNConv(10, 128)
  (conv2): GCNConv(128, 128)
  (conv3): GCNConv(128, 128)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [10]:
from src.dataset import create_graph
sample_data = create_graph(df['smiles'][0], df['gap'][0])
print(sample_data)

Data(x=[5, 5], edge_index=[2, 8], y=[1])


In [11]:
from src.train import train_model

model = train_model(
    smiles_list=df['smiles'].head(2000), 
    target_list=df['gap'].head(2000),
    batch_size=32, 
    epochs=10,
    lr=0.0005
)

Converting SMILES to Graphs...
Training on: cuda
Epoch 1/10 | Loss: 0.0168
Epoch 2/10 | Loss: 0.0022
Epoch 3/10 | Loss: 0.0019
Epoch 4/10 | Loss: 0.0017
Epoch 5/10 | Loss: 0.0016
Epoch 6/10 | Loss: 0.0015
Epoch 7/10 | Loss: 0.0015
Epoch 8/10 | Loss: 0.0014
Epoch 9/10 | Loss: 0.0014
Epoch 10/10 | Loss: 0.0013
